# 🗞️ Türkçe Haber Sınıflandırma — DistilBERT Fine-Tuning

**Veri Seti:** `batubayk/TR-News`  
**Model:** `dbmdz/distilbert-base-turkish-cased`  
**Görev:** Haber metinlerini 10 ana kategoriye sınıflandırma  
**Donanım:** RTX 4070 Laptop (8 GB VRAM) için optimize edilmiştir

| Kategori | Açıklama |
|----------|----------|
| Gündem | Türkiye, siyaset, seçim |
| Dünya | Uluslararası haberler |
| Spor | Futbol, basketbol, tüm sporlar |
| Ekonomi | Finans, borsa, enerji |
| Teknoloji | Bilim & teknoloji |
| Sağlık | Sağlık haberleri |
| Kültür-Sanat | Müzik, sinema, magazin |
| Eğitim | Eğitim haberleri |
| Yaşam | Seyahat, moda, astroloji |
| Otomobil | Otomobil & savunma |

## 0. ⚡ PyTorch CUDA Kurulumu (İlk Çalıştırmada Bir Kez Yap)

Eğer `CUDA: False` görüyorsan aşağıdaki hücreyi çalıştır ve kernel'i yeniden başlat.

In [ ]:
import torch
if not torch.cuda.is_available():
    print("⚠️  CUDA bulunamadı! GPU sürücüsü kuruluyorsa PyTorch CUDA versiyonunu yükle:")
    print()
    print("pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124")
    print()
    print("Kurulumdan sonra kernel'i yeniden başlat (Kernel → Restart).")
else:
    print(f"✅ CUDA aktif: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Kurulum ve Import

In [ ]:
import sys
import os
import csv
import json
import time
import random
import logging
import gc
import warnings
import contextlib
import platform
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.amp import GradScaler, autocast
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import (
    classification_report, accuracy_score,
    f1_score, precision_score, recall_score, confusion_matrix,
)
import matplotlib.pyplot as plt

try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("⚠️  pandas bulunamadı — error_analysis.csv atlanacak")
    print("   Kurmak için: pip install pandas")

IS_COLAB = "google.colab" in sys.modules
IS_WINDOWS = platform.system() == "Windows"

logging.basicConfig(level=logging.WARNING, format="%(levelname)s - %(message)s")
logger = logging.getLogger("news_train")

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU     : {torch.cuda.get_device_name(0)}")
print(f"Platform: {platform.system()}")
print(f"Ortam   : {'Google Colab' if IS_COLAB else 'Yerel'}")


## 2. Ayarlar (RTX 4070 Laptop 8 GB için Optimize)

In [ ]:
# ── Dizin ayarları ────────────────────────────────────────────────────
if IS_COLAB:
    BASE_DIR = Path("/content/news_project")
    BASE_DIR.mkdir(exist_ok=True)
else:
    # Kendi dizinine göre güncelle
    BASE_DIR = Path(r"C:\Users\LENOVO\Desktop\Aktif Projeler\news")

DATA_DIR  = BASE_DIR / "data"
MODEL_DIR = BASE_DIR / "model"
BEST_DIR  = MODEL_DIR / "best"
DATA_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
BEST_DIR.mkdir(parents=True, exist_ok=True)

# ── Colab'da veri setini HuggingFace'ten indir ────────────────────────
if IS_COLAB and not (DATA_DIR / "train.csv").exists():
    from datasets import load_dataset
    print("📥 TR-News veri seti indiriliyor...")
    ds = load_dataset("batubayk/TR-News")
    for split in ["train", "validation", "test"]:
        ds[split].to_pandas().to_csv(DATA_DIR / f"{split}.csv", index=False)
        print(f"  ✓ {split}.csv ({len(ds[split]):,} satır)")

# ── Device ────────────────────────────────────────────────────────────
cuda_available = torch.cuda.is_available()
device = torch.device("cuda:0") if cuda_available else torch.device("cpu")

if cuda_available:
    torch.cuda.empty_cache()
    torch.backends.cudnn.enabled   = True
    torch.backends.cudnn.benchmark = True

# ── Hiperparametreler — RTX 4070 Laptop 8 GB ──────────────────────────
MODEL_NAME    = "dbmdz/distilbert-base-turkish-cased"
MAX_LEN       = 256         # Haber başlık + özet için yeterli
BATCH_SIZE    = 32 if cuda_available else 8   # 8 GB VRAM'de güvenli
EPOCHS        = 4           # 3 yerine 4 — daha iyi yakınsama
LR            = 2e-5
WEIGHT_DECAY  = 0.01
WARMUP_RATIO  = 0.06        # Sabit adım yerine oran kullan
MAX_GRAD_NORM = 1.0
GRAD_ACCUM    = 2           # Efektif batch = 32 × 2 = 64
LABEL_SMOOTH  = 0.05        # Label smoothing for regularization
USE_AMP       = cuda_available  # Mixed Precision (float16) — RTX 4070 destekler
SEED          = 42

# ── num_workers — Windows'ta 0 olmalı ─────────────────────────────────
NUM_WORKERS = 0 if IS_WINDOWS else (4 if cuda_available else 0)

# ── Seed ──────────────────────────────────────────────────────────────
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)
csv.field_size_limit(10**7)

# ── Sistem bilgisi ────────────────────────────────────────────────────
print("=" * 70)
print("🖥️  SİSTEM BİLGİSİ")
print("=" * 70)
print(f"Device       : {device}")
if cuda_available:
    props = torch.cuda.get_device_properties(0)
    print(f"GPU          : {props.name}")
    print(f"VRAM         : {props.total_memory / 1024**3:.1f} GB")
print(f"Mixed Prec.  : {USE_AMP}")
print(f"num_workers  : {NUM_WORKERS}")
print()
print(f"Model        : {MODEL_NAME}")
print(f"Batch Size   : {BATCH_SIZE}  (Grad Accum: {GRAD_ACCUM} → Efektif: {BATCH_SIZE * GRAD_ACCUM})")
print(f"Max Len      : {MAX_LEN}")
print(f"Epochs       : {EPOCHS}")
print(f"LR           : {LR}")
print(f"Label Smooth : {LABEL_SMOOTH}")
print("=" * 70)

## 3. Kategori Eşleme (122 → 10)

In [ ]:
CATEGORY_MAP = {
    # Gündem / Siyaset
    "Türkiye": "Gündem", "turkiye": "Gündem", "Gündem": "Gündem",
    "siyaset": "Gündem", "Polemik": "Gündem",
    "2019 Yerel Seçim": "Gündem", "23 Haziran seçimleri": "Gündem",
    "secim_2015": "Gündem", "secim": "Gündem", "Seçim": "Gündem",
    "Seçim 2015": "Gündem", "yerel_yonetimler": "Gündem",
    "H. Bunu Konuşuyor": "Gündem", "15 Temmuz davaları": "Gündem",
    "Haberin Var Mı?": "Gündem", "Kadına Şiddet": "Gündem",
    "sokak": "Gündem", "Ortak Gelecek": "Gündem",

    # Dünya
    "Dünya": "Dünya", "dunya": "Dünya", "Dünyadan": "Dünya",
    "BBC": "Dünya", "english": "Dünya",

    # Spor
    "Spor": "Spor", "spor": "Spor",
    "futbol": "Spor", "Futbol": "Spor",
    "basketbol": "Spor", "Basketbol": "Spor",
    "Voleybol": "Spor", "Tenis": "Spor", "Hentbol": "Spor",
    "Motor Sporları": "Spor", "Olimpiyat": "Spor",
    "diger_sporlar": "Spor", "İddaa": "Spor",
    "EURO 2016": "Spor", "2016 Avrupa Şampiyonası": "Spor",
    "2018 Dünya Kupası": "Spor", "Dünya Kupası 2018": "Spor",
    "2016 Rio Olimpiyatları": "Spor", "foto_spor": "Spor",
    "2018 DK Tarihçe": "Spor",

    # Ekonomi
    "Ekonomi": "Ekonomi", "ekonomi": "Ekonomi",
    "Makro Ekonomi": "Ekonomi", "Para": "Ekonomi",
    "Emlak": "Ekonomi", "Enerji": "Ekonomi",
    "Döviz": "Ekonomi", "Borsa": "Ekonomi", "Altın": "Ekonomi",
    "Sigorta": "Ekonomi", "Sosyal Güvenlik": "Ekonomi",
    "Bitcoin": "Ekonomi", "Girişimcilik": "Ekonomi",
    "Akdeniz Ekonomi Forumu": "Ekonomi",
    "Ege Ekonomik Forum 2017": "Ekonomi",
    # "AIRPORT" çıkarıldı — anlamlı eşleşme değil

    # Teknoloji & Bilim
    "Teknoloji": "Teknoloji", "bilim_ve_teknoloji": "Teknoloji",
    "uzay": "Teknoloji",

    # Sağlık
    "Sağlık": "Sağlık", "saglik": "Sağlık",

    # Kültür & Sanat & Magazin
    "Sanat": "Kültür-Sanat", "Kültür-Sanat": "Kültür-Sanat",
    "kultur-sanat": "Kültür-Sanat", "foto_kultur_sanat": "Kültür-Sanat",
    "Müzik": "Kültür-Sanat", "konser": "Kültür-Sanat",
    "Televizyon": "Kültür-Sanat", "sinema": "Kültür-Sanat",
    "kitap": "Kültür-Sanat", "Biyografi": "Kültür-Sanat",
    "Fiskos": "Kültür-Sanat", "Magazin": "Kültür-Sanat",
    "Cemiyet Hayatı": "Kültür-Sanat", "Medya": "Kültür-Sanat",
    "Röportajlar": "Kültür-Sanat", "Özel Röportajlar": "Kültür-Sanat",
    "Özel": "Kültür-Sanat", "etkinlik": "Kültür-Sanat",

    # Eğitim
    "Eğitim": "Eğitim", "egitim": "Eğitim",

    # Yaşam
    "Yaşam": "Yaşam", "yasam": "Yaşam", "İş-Yaşam": "Yaşam",
    "calisma_yasami": "Yaşam", "Seyahat": "Yaşam", "gezi": "Yaşam",
    "Turizm": "Yaşam", "cevre": "Yaşam",
    "surdurulebilir_yasam": "Yaşam", "Ramazan": "Yaşam",
    "Tarifler": "Yaşam", "Yemek Yapma": "Yaşam",
    "Nasıl Yapılır": "Yaşam", "Alışveriş": "Yaşam",
    "Güvenli Alışveriş": "Yaşam", "Moda": "Yaşam", "moda": "Yaşam",
    "Evde Ek İş": "Yaşam", "Sıfır Atık": "Yaşam",
    "Merak Edilenler": "Yaşam", "Nedir": "Yaşam",
    "Astroloji": "Yaşam", "astroloji": "Yaşam",
    "Şipşak": "Yaşam", "Takılın": "Yaşam",
    "MOTY 2019": "Yaşam",

    # Otomobil
    "Otomobil": "Otomobil", "otomobil": "Otomobil",
    # "Savunma Sanayi" → Gündem'e taşındı (otomobil ile alakası yok)
    "Savunma Sanayi": "Gündem",
}

SKIP_TOPICS = {"", "diger", "Diğer", "General", "Yazı Dizisi",
               "yazi_dizileri", "pazar_yazilari",
               "cumhuriyet_ege", "cumhuriyet_pazar",
               "AIRPORT"}  # anlamlı kategori değil

print(f"Eşleme tablosundaki ham etiket sayısı: {len(CATEGORY_MAP)}")
print(f"Ana kategori sayısı: {len(set(CATEGORY_MAP.values()))}")

## 4. Veri Yükleme ve Analiz

In [ ]:
def load_csv(path: Path) -> List[Dict]:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        reader = csv.DictReader(f)
        for row in reader:
            topic = row.get("topic", "").strip()
            if topic in SKIP_TOPICS:
                continue
            category = CATEGORY_MAP.get(topic)
            if category is None:
                continue
            title    = (row.get("title")    or "").strip()
            abstract = (row.get("abstract") or "").strip()
            text = f"{title}. {abstract}" if abstract else title
            if len(text) < 10:
                continue
            rows.append({"text": text, "category": category})
    return rows

train_rows = load_csv(DATA_DIR / "train.csv")
val_rows   = load_csv(DATA_DIR / "validation.csv")
test_rows  = load_csv(DATA_DIR / "test.csv")

print(f"Train : {len(train_rows):>7,}")
print(f"Val   : {len(val_rows):>7,}")
print(f"Test  : {len(test_rows):>7,}")
print(f"Toplam: {len(train_rows)+len(val_rows)+len(test_rows):>7,}")

In [ ]:
# Kategori dağılımı ve label map
categories = sorted(set(r["category"] for r in train_rows))
cat2id = {c: i for i, c in enumerate(categories)}
id2cat = {i: c for c, i in cat2id.items()}
num_labels = len(categories)

train_dist = Counter(r["category"] for r in train_rows)
val_dist   = Counter(r["category"] for r in val_rows)
test_dist  = Counter(r["category"] for r in test_rows)

print(f"\nKategori sayısı: {num_labels}")
print(f"{'Kategori':<16} {'Train':>8} {'Val':>7} {'Test':>7} {'Train %':>8}")
print("-" * 50)
total_train = len(train_rows)
for c in categories:
    pct = train_dist[c] / total_train * 100
    print(f"{c:<16} {train_dist[c]:>8,} {val_dist[c]:>7,} {test_dist[c]:>7,} {pct:>7.1f}%")

# Label map kaydet
label_map = {"cat2id": cat2id, "id2cat": {str(k): v for k, v in id2cat.items()}}
with open(MODEL_DIR / "label_map.json", "w", encoding="utf-8") as f:
    json.dump(label_map, f, ensure_ascii=False, indent=2)
print("\n✓ label_map.json kaydedildi")

## 5. Dataset, Class Weights ve DataLoader

In [ ]:
class NewsDataset(Dataset):
    def __init__(self, texts: List[str], labels: List[int], tokenizer, max_len: int):
        self.texts     = texts
        self.labels    = labels
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self) -> int:
        return len(self.texts)

    def __getitem__(self, idx: int) -> Dict:
        enc = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         torch.tensor(self.labels[idx], dtype=torch.long),
        }


print("Tokenizer yükleniyor...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_texts  = [r["text"]     for r in train_rows]
train_labels = [cat2id[r["category"]] for r in train_rows]
val_texts    = [r["text"]     for r in val_rows]
val_labels   = [cat2id[r["category"]] for r in val_rows]
test_texts   = [r["text"]     for r in test_rows]
test_labels  = [cat2id[r["category"]] for r in test_rows]

train_ds = NewsDataset(train_texts, train_labels, tokenizer, MAX_LEN)
val_ds   = NewsDataset(val_texts,   val_labels,   tokenizer, MAX_LEN)
test_ds  = NewsDataset(test_texts,  test_labels,  tokenizer, MAX_LEN)

# ── Sınıf dengesizliği için WeightedRandomSampler ────────────────────
# Gündem (122K) vs Otomobil (1.5K) dengesizliğini giderir
class_counts = np.array([train_dist[id2cat[i]] for i in range(num_labels)])
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_labels  # normalize
sample_weights = np.array([class_weights[label] for label in train_labels])
sampler = WeightedRandomSampler(
    weights=torch.from_numpy(sample_weights).float(),
    num_samples=len(train_ds),
    replacement=True
)

# ── DataLoader — num_workers ve pin_memory ile optimize ──────────────
train_loader = DataLoader(
    train_ds, batch_size=BATCH_SIZE,
    sampler=sampler,             # shuffle=True yerine weighted sampler
    num_workers=NUM_WORKERS,
    pin_memory=cuda_available,   # GPU transferini hızlandırır
    persistent_workers=(NUM_WORKERS > 0)
)
val_loader = DataLoader(
    val_ds, batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=cuda_available,
    persistent_workers=(NUM_WORKERS > 0)
)
test_loader = DataLoader(
    test_ds, batch_size=BATCH_SIZE * 2,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=cuda_available,
    persistent_workers=(NUM_WORKERS > 0)
)

print(f"✓ Train batches : {len(train_loader):,}")
print(f"✓ Val batches   : {len(val_loader):,}")
print(f"✓ Test batches  : {len(test_loader):,}")
print(f"✓ num_workers   : {NUM_WORKERS}")
print(f"✓ pin_memory    : {cuda_available}")
print(f"\nSınıf ağırlıkları:")
for i, c in id2cat.items():
    print(f"  [{i}] {c:<16} weight={class_weights[i]:.4f}  samples={class_counts[i]:,}")

## 6. Model Yükleme

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=num_labels
)
model.to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Toplam parametre  : {total_params:,}")
print(f"Eğitilebilir      : {trainable_params:,}")
print(f"Model hazır       : {device}")

if cuda_available:
    used = torch.cuda.memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"VRAM kullanımı    : {used:.2f} / {total:.1f} GB")

## 7. Loss Function, Optimizer, Scheduler ve Checkpoint

In [ ]:
# ── Label Smoothing Cross Entropy Loss ───────────────────────────────
class LabelSmoothingCrossEntropy(nn.Module):
    """Cross entropy with label smoothing for regularization."""
    def __init__(self, smoothing: float = 0.0):
        super().__init__()
        self.smoothing = smoothing
        self.confidence = 1.0 - smoothing
    
    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        log_probs = torch.log_softmax(logits, dim=-1)
        nll_loss = -log_probs.gather(dim=-1, index=targets.unsqueeze(1)).squeeze(1)
        smooth_loss = -log_probs.mean(dim=-1)
        loss = self.confidence * nll_loss + self.smoothing * smooth_loss
        return loss.mean()


@dataclass
class TrainingState:
    epoch: int = 0
    global_step: int = 0
    best_f1: float = 0.0
    failed_batches: int = 0
    history: Dict = field(default_factory=lambda: {
        "train_loss": [], "val_loss": [], "val_acc": [],
        "val_f1": [], "val_precision": [], "val_recall": []
    })


class EarlyStopping:
    def __init__(self, patience: int = 3, min_delta: float = 1e-4):
        self.patience  = patience
        self.min_delta = min_delta
        self.counter   = 0
        self.best      = 0.0

    def __call__(self, f1: float) -> bool:
        if f1 > self.best + self.min_delta:
            self.best    = f1
            self.counter = 0
            return False
        self.counter += 1
        return self.counter >= self.patience


class FullCheckpoint:
    """Full checkpoint with model, optimizer, scheduler and training state."""
    
    def __init__(self, best_dir: Path, checkpoint_dir: Path):
        self.best_dir = Path(best_dir)
        self.checkpoint_dir = Path(checkpoint_dir)
        self.best_f1 = 0.0
        self.best_dir.mkdir(parents=True, exist_ok=True)
        self.checkpoint_dir.mkdir(parents=True, exist_ok=True)
    
    def save_epoch(self, epoch: int, model, tokenizer, optimizer, scheduler, 
                   train_state: TrainingState, metrics: Dict):
        """Save full checkpoint at end of epoch for resume capability."""
        epoch_dir = self.checkpoint_dir / f"checkpoint-epoch-{epoch}"
        epoch_dir.mkdir(parents=True, exist_ok=True)
        
        # Save model and tokenizer
        model.save_pretrained(str(epoch_dir))
        tokenizer.save_pretrained(str(epoch_dir))
        
        # Save training state (optimizer, scheduler, history)
        training_state = {
            "epoch": epoch,
            "global_step": train_state.global_step,
            "best_f1": train_state.best_f1,
            "history": train_state.history,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "metrics": metrics,
        }
        torch.save(training_state, epoch_dir / "training_state.pt")
        logger.info(f"Checkpoint saved: {epoch_dir}")
    
    def maybe_save_best(self, model, tokenizer, metrics: Dict, info: Dict) -> bool:
        """Save best model if F1 improved."""
        f1 = metrics.get("val_f1", 0.0)
        if f1 <= self.best_f1:
            return False
        try:
            self.best_f1 = f1
            model.save_pretrained(str(self.best_dir))
            tokenizer.save_pretrained(str(self.best_dir))
            meta = {"metrics": metrics, "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"), **info}
            with open(self.best_dir / "checkpoint_info.json", "w", encoding="utf-8") as f:
                json.dump(meta, f, indent=2, ensure_ascii=False)
            return True
        except Exception as e:
            logger.error(f"Checkpoint hatası: {e}")
            return False
    
    @staticmethod
    def resume(checkpoint_path: Path, model, optimizer, scheduler) -> Tuple[int, Dict]:
        """Resume training from checkpoint."""
        state_path = checkpoint_path / "training_state.pt"
        if not state_path.exists():
            raise FileNotFoundError(f"Training state not found: {state_path}")
        
        state = torch.load(state_path, map_location="cpu")
        optimizer.load_state_dict(state["optimizer_state_dict"])
        scheduler.load_state_dict(state["scheduler_state_dict"])
        
        logger.info(f"Resumed from epoch {state['epoch']}, best_f1={state['best_f1']:.4f}")
        return state["epoch"], state["history"]


# ── Loss Function with Label Smoothing ───────────────────────────────
criterion = LabelSmoothingCrossEntropy(smoothing=LABEL_SMOOTH)

# ── Optimizer ────────────────────────────────────────────────────────
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR, weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999), eps=1e-8
)

# ── Scheduler ────────────────────────────────────────────────────────
total_steps   = (len(train_loader) // GRAD_ACCUM) * EPOCHS
warmup_steps  = int(total_steps * WARMUP_RATIO)
scheduler     = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

# ── Mixed Precision Scaler (new API) ─────────────────────────────────
scaler = GradScaler("cuda") if USE_AMP else None

# ── Callback'ler ─────────────────────────────────────────────────────
early_stop  = EarlyStopping(patience=3)
checkpoint  = FullCheckpoint(BEST_DIR, MODEL_DIR)
train_state = TrainingState()

print(f"Toplam adım    : {total_steps:,}")
print(f"Warmup adımı   : {warmup_steps:,} ({WARMUP_RATIO*100:.0f}%)")
print(f"Label Smooth   : {LABEL_SMOOTH}")
print(f"Mixed Prec.    : {USE_AMP}")
print(f"Grad Accum     : {GRAD_ACCUM}")
print("✓ Hazır")

## 8. Eğitim Döngüsü

In [ ]:
def run_epoch(model, loader, optimizer, scheduler, scaler, criterion, device,
              grad_accum: int, epoch: int, epochs: int) -> Tuple[float, int]:
    """Tek bir eğitim epoch'u. Mixed precision + gradient accumulation destekli."""
    model.train()
    total_loss    = 0.0
    valid_batches = 0
    failed        = 0
    optimizer.zero_grad()

    n = len(loader)
    t0 = time.time()

    for step, batch in enumerate(loader):
        try:
            input_ids      = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels         = batch["labels"].to(device, non_blocking=True)

            # Forward — mixed precision (new API)
            ctx = autocast("cuda", dtype=torch.float16) if scaler else contextlib.nullcontext()
            with ctx:
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                # Use custom loss with label smoothing
                loss = criterion(outputs.logits, labels) / grad_accum

            if not torch.isfinite(loss):
                logger.warning(f"NaN/Inf loss step {step}, atlanıyor")
                failed += 1
                optimizer.zero_grad()
                continue

            # Backward
            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            # Optimizer adımı (her grad_accum batch'te bir)
            if (step + 1) % grad_accum == 0 or (step + 1) == n:
                if scaler:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
                    optimizer.step()
                scheduler.step()
                optimizer.zero_grad()

            total_loss    += loss.item() * grad_accum  # geri al normalize
            valid_batches += 1

            # İlerleme
            if (step + 1) % max(1, n // 10) == 0 or (step + 1) == n:
                pct     = (step + 1) / n * 100
                elapsed = time.time() - t0
                eta     = elapsed / (step + 1) * (n - step - 1)
                avg_l   = total_loss / valid_batches
                lr_now  = optimizer.param_groups[0]["lr"]
                print(f"  Epoch {epoch}/{epochs} | {pct:5.1f}% | "
                      f"loss={avg_l:.4f} | lr={lr_now:.2e} | "
                      f"ETA={int(eta//60)}m{int(eta%60):02d}s", flush=True)

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                logger.warning(f"OOM step {step}, cache temizleniyor")
                torch.cuda.empty_cache(); gc.collect()
            else:
                logger.error(f"RuntimeError step {step}: {e}")
            failed += 1
            optimizer.zero_grad()

    avg_loss = total_loss / valid_batches if valid_batches > 0 else 0.0
    return avg_loss, failed


def evaluate(model, loader, criterion, device) -> Dict[str, float]:
    """Validation / Test metriklerini hesapla."""
    model.eval()
    total_loss = 0.0
    preds_all, labels_all = [], []
    batches = 0

    with torch.no_grad():
        for batch in loader:
            try:
                input_ids      = batch["input_ids"].to(device, non_blocking=True)
                attention_mask = batch["attention_mask"].to(device, non_blocking=True)
                labels         = batch["labels"].to(device, non_blocking=True)

                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss = criterion(outputs.logits, labels)

                if torch.isfinite(loss):
                    total_loss += loss.item()
                    batches    += 1

                preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
                preds_all.extend(preds)
                labels_all.extend(labels.cpu().numpy())
            except Exception as e:
                logger.warning(f"Eval batch hatası: {e}")

    if not preds_all:
        return dict(val_loss=0, val_acc=0, val_f1=0, val_precision=0, val_recall=0)

    p = np.array(preds_all, dtype=np.int32)
    l = np.array(labels_all, dtype=np.int32)
    return {
        "val_loss":      total_loss / batches if batches > 0 else 0.0,
        "val_acc":       float(accuracy_score(l, p)),
        "val_f1":        float(f1_score(l, p, average="macro", zero_division=0)),
        "val_precision": float(precision_score(l, p, average="macro", zero_division=0)),
        "val_recall":    float(recall_score(l, p, average="macro", zero_division=0)),
    }


# ════════════════════════════════════════════════════════════════════
# ANA EĞİTİM DÖNGÜSÜ
# ════════════════════════════════════════════════════════════════════

history = train_state.history

logger.info("🚀 Eğitim başlıyor...")
logger.info(f"  Device: {device} | AMP: {USE_AMP} | Batch: {BATCH_SIZE} × Accum {GRAD_ACCUM}")

t_start = time.time()

try:
    for epoch in range(1, EPOCHS + 1):
        print(f"\n{'═'*65}")
        print(f"EPOCH {epoch}/{EPOCHS}")
        print(f"{'═'*65}")

        train_loss, failed = run_epoch(
            model, train_loader, optimizer, scheduler, scaler, criterion,
            device, GRAD_ACCUM, epoch, EPOCHS
        )
        train_state.failed_batches += failed

        print(f"  → Val tahminleri hesaplanıyor...", flush=True)
        val_metrics = evaluate(model, val_loader, criterion, device)

        # Update training state
        train_state.epoch = epoch
        if val_metrics["val_f1"] > train_state.best_f1:
            train_state.best_f1 = val_metrics["val_f1"]

        # Kayıt
        for k in history:
            v = train_loss if k == "train_loss" else val_metrics.get(k, 0.0)
            history[k].append(v)

        elapsed = time.time() - t_start
        print(f"\n  📊 Sonuçlar:")
        print(f"     Train Loss : {train_loss:.4f}")
        print(f"     Val Loss   : {val_metrics['val_loss']:.4f}")
        print(f"     Val Acc    : {val_metrics['val_acc']:.4f}")
        print(f"     Val F1     : {val_metrics['val_f1']:.4f}  ⭐")
        print(f"     Süre       : {int(elapsed//60)}m{int(elapsed%60):02d}s")
        if failed > 0:
            print(f"     ⚠️  Failed  : {failed} batch")

        # Save epoch checkpoint (for resume)
        checkpoint.save_epoch(epoch, model, tokenizer, optimizer, scheduler, 
                              train_state, val_metrics)

        # Save best model
        saved = checkpoint.maybe_save_best(model, tokenizer, val_metrics, {"epoch": epoch})
        if saved:
            print(f"     ✅ En iyi model kaydedildi! F1={checkpoint.best_f1:.4f}")

        # GPU bellek raporu
        if cuda_available:
            used  = torch.cuda.memory_allocated(0)  / 1024**3
            total = torch.cuda.get_device_properties(0).total_memory / 1024**3
            print(f"     🎮 VRAM     : {used:.2f} / {total:.1f} GB")

        # Early stopping
        if early_stop(val_metrics["val_f1"]):
            print(f"\n  ⛔ Early stopping tetiklendi (patience={early_stop.patience})")
            break

    total_time = time.time() - t_start
    print(f"\n{'═'*65}")
    print(f"✅ EĞİTİM TAMAMLANDI")
    print(f"   Toplam Süre   : {int(total_time//60)}m{int(total_time%60):02d}s")
    print(f"   En İyi F1     : {checkpoint.best_f1:.4f}")
    print(f"   Best Model    : {BEST_DIR}")
    print(f"{'═'*65}")

except KeyboardInterrupt:
    elapsed = time.time() - t_start
    print(f"\n⚠️  Eğitim kullanıcı tarafından durduruldu ({int(elapsed//60)}m{int(elapsed%60):02d}s)")

except Exception as e:
    logger.error(f"❌ Kritik hata: {e}", exc_info=True)
    raise

## 9. Eğitim Grafikleri

In [ ]:
n_epochs = len(history.get("train_loss", []))

if n_epochs == 0:
    print("⚠️  Henüz eğitim tamamlanmadı — grafik çizilemiyor.")
    print("   Önce Cell 8 (Eğitim Döngüsü) hücresini çalıştırın.")
else:
    x = range(1, n_epochs + 1)

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle("Eğitim Grafikleri", fontsize=14, fontweight="bold")

    # Loss
    axes[0].plot(x, history["train_loss"], "b-o", label="Train")
    axes[0].plot(x, history["val_loss"],   "r-o", label="Val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy
    axes[1].plot(x, history["val_acc"], "g-o")
    axes[1].set_title("Validation Accuracy")
    axes[1].set_xlabel("Epoch")
    if max(history["val_acc"]) > 0.5:
        axes[1].set_ylim(0.5, 1.0)
    axes[1].grid(True, alpha=0.3)

    # F1
    axes[2].plot(x, history["val_f1"], "m-o")
    axes[2].set_title("Validation F1-macro")
    axes[2].set_xlabel("Epoch")
    if max(history["val_f1"]) > 0.5:
        axes[2].set_ylim(0.5, 1.0)
    axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(str(BASE_DIR / "training_curves.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ Grafik kaydedildi: training_curves.png")


## 10. Test Sonuçları

In [ ]:
# ── En iyi modeli yükle (BEST_DIR → checkpoint fallback) ─────────────
def _load_best_model(best_dir: Path, model_dir: Path, device):
    """BEST_DIR yoksa en son checkpoint'tan yükle."""
    if (best_dir / "config.json").exists():
        print(f"✓ Best model: {best_dir}")
        return AutoModelForSequenceClassification.from_pretrained(str(best_dir)).to(device)
    # Checkpoint fallback
    ckpts = sorted(
        [d for d in model_dir.iterdir()
         if d.is_dir() and d.name.startswith("checkpoint-epoch-") and (d / "config.json").exists()],
        key=lambda d: int(d.name.split("-")[-1]),
        reverse=True,
    )
    if ckpts:
        print(f"✓ Fallback checkpoint: {ckpts[0]}")
        return AutoModelForSequenceClassification.from_pretrained(str(ckpts[0])).to(device)
    raise FileNotFoundError(
        "Kaydedilmis model bulunamadi!
"
        "Oncelıkle Cell 8 (Egitim Dongusu) hucresini calistirin."
    )

print("En iyi model yukleniyor...")
best_model = _load_best_model(BEST_DIR, MODEL_DIR, device)
best_model.eval()

test_metrics = evaluate(best_model, test_loader, criterion, device)
target_names = [id2cat[i] for i in range(num_labels)]

# Detaylı rapor için predictions tekrar topla
all_preds, all_labels, all_texts_test = [], [], []
_bs = BATCH_SIZE * 2
with torch.no_grad():
    for i, batch in enumerate(test_loader):
        input_ids      = batch["input_ids"].to(device, non_blocking=True)
        attention_mask = batch["attention_mask"].to(device, non_blocking=True)
        labels         = batch["labels"]
        outputs = best_model(input_ids=input_ids, attention_mask=attention_mask)
        preds = torch.argmax(outputs.logits, dim=-1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        # Karşılık gelen metinleri ekle
        start_idx = i * _bs
        end_idx   = min(start_idx + _bs, len(test_texts))
        all_texts_test.extend(test_texts[start_idx:end_idx])

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

print("
Test Classification Report")
print("=" * 65)
print(classification_report(all_labels, all_preds, target_names=target_names, digits=4))
test_acc = float(accuracy_score(all_labels, all_preds))
test_f1  = float(f1_score(all_labels, all_preds, average="macro"))
print(f"Test Accuracy : {test_acc:.4f}")
print(f"Test F1-macro : {test_f1:.4f}")


## 11. Confusion Matrix

In [ ]:
if "all_preds" not in dir() or len(all_preds) == 0:
    print("⚠️  Önce Cell 10 (Test Sonuçları) hücresini çalıştırın.")
else:
    cm = confusion_matrix(all_labels, all_preds)

    fig, ax = plt.subplots(figsize=(11, 9))
    im = ax.imshow(cm, cmap="Blues")

    ax.set_xticks(range(num_labels))
    ax.set_yticks(range(num_labels))
    ax.set_xticklabels(target_names, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(target_names, fontsize=9)
    ax.set_xlabel("Tahmin")
    ax.set_ylabel("Gerçek")
    ax.set_title("Confusion Matrix (Test Set)")

    for i in range(num_labels):
        for j in range(num_labels):
            color = "white" if cm[i, j] > cm.max() / 2 else "black"
            ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                    color=color, fontsize=8)

    plt.colorbar(im)
    plt.tight_layout()
    plt.savefig(str(BASE_DIR / "confusion_matrix.png"), dpi=150, bbox_inches="tight")
    plt.show()
    print("✓ confusion_matrix.png kaydedildi")


## 12. Error Analysis

In [ ]:
if "all_preds" not in dir() or len(all_preds) == 0:
    print("⚠️  Önce Cell 10 (Test Sonuçları) hücresini çalıştırın.")
else:
    errors = []
    for i, (pred, label) in enumerate(zip(all_preds, all_labels)):
        if pred != label:
            errors.append({
                "text":       all_texts_test[i][:500] if i < len(all_texts_test) else "",
                "true_label": id2cat[int(label)],
                "predicted":  id2cat[int(pred)],
            })

    print(f"Toplam yanlış tahmin: {len(errors):,} / {len(all_preds):,} ({len(errors)/len(all_preds)*100:.1f}%)")

    if HAS_PANDAS:
        error_df = pd.DataFrame(errors)
        error_df.to_csv(BASE_DIR / "error_analysis.csv", index=False, encoding="utf-8")
        print(f"✓ error_analysis.csv kaydedildi ({len(errors):,} satır)")
    else:
        print("pandas yok — CSV atlandı")

    # En çok karıştırılan kategori çiftleri
    from collections import Counter as _Counter
    confusion_pairs = _Counter([(e["true_label"], e["predicted"]) for e in errors])
    print("
En çok karıştırılan kategori çiftleri:")
    for (true, pred), count in confusion_pairs.most_common(10):
        print(f"  {true:<16} → {pred:<16} : {count:>4}")


## 13. Hızlı Test — Kendi Cümlelerinle Dene

In [ ]:
if "best_model" not in dir():
    print("⚠️  Önce Cell 10 (Test Sonuçları) hücresini çalıştırın.")
else:
    def predict(text: str) -> Tuple[str, float, Dict[str, float]]:
        """Tek bir metni sınıflandır. Kategori + güven + tüm olasılıklar."""
        best_model.eval()
        enc = tokenizer(
            text, max_length=MAX_LEN,
            padding="max_length", truncation=True, return_tensors="pt"
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        with torch.no_grad():
            logits = best_model(**enc).logits
        probs   = torch.softmax(logits, dim=-1).squeeze()
        pred_id = probs.argmax().item()
        all_probs = {id2cat[i]: round(probs[i].item(), 4) for i in range(num_labels)}
        return id2cat[pred_id], probs[pred_id].item(), all_probs

    test_sentences = [
        "Galatasaray derbide Fenerbahçe'yi 3-1 mağlup etti",
        "Dolar kuru 35 TL'yi aştı, Merkez Bankası faiz kararını açıkladı",
        "Yapay zeka alanında yeni bir çip geliştirildi",
        "Cumhurbaşkanı yeni kabineyi açıkladı",
        "İstanbul'da yeni metro hattı açıldı vatandaşlar memnun",
        "Grip salgınında vaka sayıları artıyor, uzmanlar uyarıyor",
        "Cannes Film Festivali'nde Türk yönetmene ödül",
        "YKS başvuruları bugün sona eriyor",
        "BMW yeni elektrikli SUV modelini tanıttı",
        "Rusya-Ukrayna savaşında yeni gelişmeler",
    ]

    print("Tahmin Sonuçları")
    print("=" * 72)
    for s in test_sentences:
        cat, conf, _ = predict(s)
        print(f"  [{cat:<14}] (%{conf*100:5.1f})  {s}")


## 14. Model Bilgisi Kaydet

In [ ]:
# test_acc ve test_f1 yoksa (Cell 10 çalışmadıysa) sıfır kullan
_test_acc = globals().get("test_acc", 0.0)
_test_f1  = globals().get("test_f1",  0.0)

training_info = {
    "model_name":      MODEL_NAME,
    "num_labels":      num_labels,
    "categories":      categories,
    "max_len":         MAX_LEN,
    "batch_size":      BATCH_SIZE,
    "grad_accum":      GRAD_ACCUM,
    "effective_batch": BATCH_SIZE * GRAD_ACCUM,
    "epochs":          EPOCHS,
    "learning_rate":   LR,
    "label_smoothing": LABEL_SMOOTH,
    "use_amp":         USE_AMP,
    "best_val_f1":     checkpoint.best_f1 if "checkpoint" in globals() else 0.0,
    "test_accuracy":   _test_acc,
    "test_f1_macro":   _test_f1,
    "train_samples":   len(train_rows),
    "val_samples":     len(val_rows),
    "test_samples":    len(test_rows),
    "history":         history if "history" in globals() else {},
    "device":          str(device),
    "platform":        platform.system(),
    "timestamp":       time.strftime("%Y-%m-%d %H:%M:%S"),
}

with open(MODEL_DIR / "training_info.json", "w", encoding="utf-8") as f:
    json.dump(training_info, f, ensure_ascii=False, indent=2)

print("✓ training_info.json kaydedildi")
print(f"
Sonuçlar:")
print(f"  Val F1  (best) : {training_info['best_val_f1']:.4f}")
print(f"  Test Acc       : {_test_acc:.4f}")
print(f"  Test F1-macro  : {_test_f1:.4f}")

print(f"
Model dosyaları ({MODEL_DIR}):")
for fp in sorted(MODEL_DIR.iterdir()):
    if fp.is_file():
        print(f"  {fp.name:<35} {fp.stat().st_size / 1024**2:6.1f} MB")
print(f"
Best model: {BEST_DIR}")
